# Notebook 02 · Modelado

El dataset tiene un desbalance de 19:1 (95% sin ictus / 5% con ictus). Accuracy no es una métrica válida. Usamos AUC-ROC como métrica principal y Recall como métrica secundaria para minimizar falsos negativos.

## Índice
1. Setup e importaciones  
2. Configuración de MLflow  
3. Carga y versiones del dataset  
4. Preprocesado con ColumnTransformer  
5. Pipeline de entrenamiento  
6. Función principal `run_experiment`  
7. Ejecución de experimentos  
8. Comparativa de resultados

## 01 · Setup e importaciones

Cargamos todas las dependencias necesarias. Hay dos puntos clave aquí:
* Usamos <code>imblearn.pipeline.Pipeline</code> en lugar de <code>sklearn.pipeline.Pipeline</code> — esto es fundamental porque la versión de imbalanced-learn sabe que SMOTE solo debe aplicarse durante el <code>fit()</code>, nunca durante el <code>predict()</code> ni en el test set.<br><br>
* <code>warnings.filterwarnings('ignore')</code> limpia la salida del notebook — en producción esto nunca se haría.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML — sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    recall_score,
    f1_score,
    precision_score,
    confusion_matrix,
    RocCurveDisplay
)


# ⚠️ IMPORTANTE: usar imblearn.pipeline, NO sklearn.pipeline -- imblearn/Pipeline que entiende SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# MLflow
import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings("ignore")

# ── Constantes globales ──
RANDOM_STATE = 42
TEST_SIZE    = 0.2

print("✓ Importaciones completadas")

/home/under/miniconda3/envs/py310/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


✓ Importaciones completadas


## 02 · Configuración de MLflow

Usamos MLflow para hacer tracking de experimentos. Cada vez que se entrena un modelo, MLflow guarda automáticamente los parámetros, métricas y el modelo serializado. Esto nos permite:
* Reproducir cualquier experimento anterior con exactamente los mismos parámetros
* Comparar todos los modelos del equipo en una sola tabla
* Versionar los modelos y descargar el mejor para producción

Para ver la UI ejecutar en terminal (Solo con WSL): <code>mlflow ui --backend-store-uri file:///mnt/c/Users/under/Documents/F5/3_Projects/Stroker_project/mlruns --port 5001</code> y abrir <code>http://localhost:5001</code>


In [ ]:
# ── Configuración MLflow ──
# Guardamos los experimentos en local (carpeta mlruns/)
MLFLOW_URI = "file:///mnt/c/Users/under/Documents/F5/3_Projects/Stroker_project/mlruns"

# ACUERDO DE EQUIPO: todos usamos exactamente este nombre
EXPERIMENT_NAME = "stroke_project"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"✓ MLflow configurado")

✓ MLflow configurado
  Tracking URI : file:///mnt/c/Users/under/Documents/F5/3_Projects/Stroker_project/mlruns
  Experimento  : stroke_project
  UI           : mlflow ui → http://localhost:5001


## 03 · Carga y versiones del dataset

<code>get_dataset(version)</code> centraliza toda la lógica de limpieza en un solo lugar. Si el equipo decide cambiar cómo se trata <code>work_type='children'</code>, 
solo hay que modificar esta función y todos los experimentos se actualizan automáticamente.

Ofrecemos dos versiones del dataset:
* full: todos los pacientes (incluye menores de 17 años)
* adults: solo pacientes ≥ 17 años (decisión del EDA)


In [4]:
# Carga inicial — el raw nunca se modifica
df = pd.read_csv("../data/raw/stroke_dataset.csv")
print(f"✓ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"  Stroke=1 : {df['stroke'].sum()} ({df['stroke'].mean()*100:.1f}%)")
print(f"  Stroke=0 : {(df['stroke']==0).sum()} ({(1-df['stroke'].mean())*100:.1f}%)")

✓ Dataset cargado: 4,981 filas × 11 columnas
  Stroke=1 : 248 (5.0%)
  Stroke=0 : 4733 (95.0%)


In [5]:
def get_dataset(version: str = "full") -> pd.DataFrame:
    """
    Devuelve el dataset con las limpiezas acordadas en el EDA.
    
    Parámetros
    ----------
    version : str
        'full'   → todos los pacientes
        'adults' → solo pacientes con age >= 17
    
    Retorna
    -------
    pd.DataFrame con las transformaciones aplicadas
    
    Decisiones del EDA aplicadas aquí
    ----------------------------------
    - work_type='children' → 'not_applied' (no tiene sentido laboral para menores)
    - smoking_status='Unknown' se mantiene como categoría implícita de faltante
    """
    assert version in ("full", "adults"), f"version debe ser 'full' o 'adults', recibido: {version}"
    
    df_copy = df.copy()
    
    # Decisión EDA: work_type='children' → 'not_applied'
    df_copy.loc[df_copy["work_type"] == "children", "work_type"] = "not_applied"
    
    # Decisión EDA: filtrar menores si se pide versión adultos
    if version == "adults":
        before = len(df_copy)
        df_copy = df_copy[df_copy["age"] >= 17].copy()
        print(f"  Filtro adultos: {before - len(df_copy)} filas eliminadas")
    
    print(f"✓ Dataset '{version}': {df_copy.shape[0]:,} filas")
    return df_copy

## 04 · Preprocesado con ColumnTransformer

<code>ColumnTransformer</code> aplica transformaciones distintas a cada grupo de columnas en paralelo:<br><br>

* Numéricas → StandardScaler: centra en media 0 y escala a desviación 1. Necesario para Logistic Regression (sensible a la escala). Random Forest y XGBoost no lo necesitan pero no les perjudica.
* Categóricas → OneHotEncoder: convierte categorías en columnas binarias. <code>handle_unknown='ignore'</code> evita errores si en producción llega una categoría no vista en entrenamiento.

In [ ]:
def build_preprocessor(X_train: pd.DataFrame) -> ColumnTransformer:
    """
    Construye el preprocesador sobre X_train únicamente.
    
    El fit del ColumnTransformer aprenderá media, std y categorías
    solo del conjunto de entrenamiento — nunca del test.
    
    Parámetros
    ----------
    X_train : pd.DataFrame
        Conjunto de entrenamiento (sin la columna target)
    
    Retorna
    -------
    ColumnTransformer sin fitear (el Pipeline lo fiteará en .fit())
    """
    cat_cols = X_train.select_dtypes(include="object").columns.tolist()
    num_cols = X_train.select_dtypes(exclude="object").columns.tolist()
    
    print(f"  Columnas numéricas  : {num_cols}")
    print(f"  Columnas categóricas: {cat_cols}")
    
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(),                        num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"),  cat_cols),
    ])
    
    return preprocessor

## 05 · Pipeline de entrenamiento

El Pipeline encadena pasos en orden. Cuando llamamos a <code>pipeline.fit(X_train, y_train)</code>, cada paso se ejecuta en secuencia:

> prep → [smote] → model

La magia del Pipeline de imblearn es que durante el <code>predict()</code>, SMOTE simplemente no se ejecuta — solo actúa en el <code>fit()</code>. Esto garantiza que el test nunca se modifique.


In [ ]:
def build_pipeline(model, preprocessor, use_smote: bool = False) -> Pipeline:
    """
    Construye el Pipeline de entrenamiento.
    
    Orden de pasos:
        1. prep  : ColumnTransformer (escala + encoding)
        2. smote : SMOTE (solo si use_smote=True, solo en fit)
        3. model : clasificador
        
    Parámetros
    ----------
    model        : estimador sklearn compatible
    preprocessor : ColumnTransformer ya construido
    use_smote    : bool — si True, añade SMOTE al pipeline
    """
    steps = [("prep", preprocessor)]
    
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    
    steps.append(("model", model))
    
    return Pipeline(steps)

## 06 · Función principal run_experiment

Esta función orquesta todo el flujo de un experimento:

**Métricas que logueamos en MLflow:**

* auc — métrica principal, robusta al desbalance
* recall — minimizar falsos negativos (crítico en contexto médico)
* f1 — balance entre precision y recall
* precision — de los que predecimos positivo, cuántos lo son

**Tags vs Params en MLflow:**
* mlflow.log_param() → hiperparámetros del modelo (numéricos, comparables)
* mlflow.set_tag() → metadatos del experimento (autor, dataset, versión)

In [ ]:
def run_experiment(model, model_name: str, dataset_version: str = "full", use_smote: bool = False, author: str = "Jonathan") -> dict:
    """
    Entrena un modelo, evalúa y registra todo en MLflow.
    
    Parámetros
    ----------
    model           : estimador sklearn (LogisticRegression, RandomForest, etc.)
    model_name      : str — nombre del run en MLflow
    dataset_version : str — 'full' o 'adults'
    use_smote       : bool — si True aplica SMOTE en entrenamiento
    author          : str — nombre del miembro del equipo
    
    Retorna
    -------
    dict con métricas del experimento (para comparativa al final)
    """
    print(f"\n{'='*60}")
    print(f"  {model_name} | dataset={dataset_version} | SMOTE={use_smote}")
    print(f"{'='*60}")
    
    # ── 1. Datos ──
    df_exp = get_dataset(dataset_version)
    X = df_exp.drop("stroke", axis=1)
    y = df_exp["stroke"]
    
    # Split estratificado: garantiza misma proporción de clases en train y test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        stratify=y,          # ← fundamental con datos desbalanceados
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )
    
    print(f"  Train: {len(X_train):,} filas | Test: {len(X_test):,} filas")
    print(f"  Stroke en train: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
    
    # ── 2. Pipeline ──
    preprocessor = build_preprocessor(X_train)
    pipeline = build_pipeline(model, preprocessor, use_smote)
    
    # ── 3. Entrenamiento + evaluación dentro del run de MLflow ────
    with mlflow.start_run(run_name=f"Jonathan_{model_name}_{dataset_version}_smote={use_smote}"):
        
        # Entrenamiento
        pipeline.fit(X_train, y_train)
        
        # Predicciones
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        
        # ── Métricas ──
        auc       = roc_auc_score(y_test, y_prob)
        recall    = recall_score(y_test, y_pred)
        f1        = f1_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        
        # ── Log métricas (acuerdo de equipo: estos nombres exactos) ──
        mlflow.log_metric("auc",       round(auc, 4))
        mlflow.log_metric("recall",    round(recall, 4))
        mlflow.log_metric("f1",        round(f1, 4))
        mlflow.log_metric("precision", round(precision, 4))
        
        # ── Log hiperparámetros del modelo ──
        # get_params() devuelve todos los parámetros automáticamente
        mlflow.log_params(model.get_params())
        
        # ── Tags: metadatos del experimento (no son hiperparámetros) ──
        mlflow.set_tag("author",          author)
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("use_smote",       str(use_smote))
        mlflow.set_tag("model_type",      model_name)
        
        # ── Guardar el pipeline completo (preprocesado + modelo) ──
        mlflow.sklearn.log_model(pipeline, artifact_path=model_name)
        
        # ── Output en notebook ──
        print(f"\n  AUC-ROC   : {auc:.4f}  ← métrica principal")
        print(f"  Recall    : {recall:.4f}  ← minimizar falsos negativos")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  Precision : {precision:.4f}")
        print(f"\n{classification_report(y_test, y_pred, target_names=['Sin ictus', 'Ictus'])}")
        
        print(f"  ✓ Run registrado en MLflow")
        
        return {
            "modelo":   model_name,
            "dataset":  dataset_version,
            "smote":    use_smote,
            "auc":      round(auc, 4),
            "recall":   round(recall, 4),
            "f1":       round(f1, 4),
            "precision":round(precision, 4),
        }

## 07 · Ejecución de experimentos

Aquí definimos los modelos y lanzamos los experimentos. Cada llamada a <code>run_experiment</code> genera un run independiente en MLflow con sus propias métricas, parámetros y el modelo serializado.<br><br>

Guardamos todos los resultados en una lista para generar la comparativa al final (Local).

In [ ]:
# ── Definición de modelos ────────────────────────────────────────
# class_weight='balanced' ajusta los pesos inversamente proporcional a la frecuencia de cada clase.

lr_model = LogisticRegression(
    class_weight="balanced",  # compensa el desbalance 19:1
    max_iter=1000,             # más iteraciones para datasets con escalas distintas
    random_state=RANDOM_STATE
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

print("✓ Modelos definidos")
print(f"  Logistic Regression  : {lr_model.get_params()}")
print(f"  Random Forest        : {rf_model.get_params()}")

✓ Modelos definidos
  Logistic Regression  : {'C': 1.0, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 1000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
  Random Forest        : {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}


In [ ]:
# ── Lanzamiento de experimentos ──
# Guardamos cada resultado para la comparativa final
AUTHOR = "Jonathan"

results = []

# Baseline: Logistic Regression sin SMOTE
results.append(run_experiment(lr_model, "LogisticRegression", "full",   use_smote=False, author=AUTHOR))

# Logistic Regression con SMOTE
results.append(run_experiment(lr_model, "LogisticRegression", "full",   use_smote=True,  author=AUTHOR))

# Random Forest sin SMOTE
results.append(run_experiment(rf_model, "RandomForest",       "full",   use_smote=False, author=AUTHOR))

# Random Forest con SMOTE
results.append(run_experiment(rf_model, "RandomForest",       "full",   use_smote=True,  author=AUTHOR))

# Solo adultos — ¿mejora el modelo sin los menores?
results.append(run_experiment(lr_model, "LogisticRegression", "adults", use_smote=False, author=AUTHOR))
results.append(run_experiment(rf_model, "RandomForest",       "adults", use_smote=True,  author=AUTHOR))


  LogisticRegression | dataset=full | SMOTE=False
✓ Dataset 'full': 4,981 filas
  Train: 3,984 filas | Test: 997 filas
  Stroke en train: 198 (5.0%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']


2026/04/17 11:34:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 11:34:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  AUC-ROC   : 0.8412  ← métrica principal
  Recall    : 0.8400  ← minimizar falsos negativos
  F1-score  : 0.2545
  Precision : 0.1500

              precision    recall  f1-score   support

   Sin ictus       0.99      0.75      0.85       947
       Ictus       0.15      0.84      0.25        50

    accuracy                           0.75       997
   macro avg       0.57      0.79      0.55       997
weighted avg       0.95      0.75      0.82       997

  ✓ Run registrado en MLflow

  LogisticRegression | dataset=full | SMOTE=True
✓ Dataset 'full': 4,981 filas
  Train: 3,984 filas | Test: 997 filas
  Stroke en train: 198 (5.0%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']


2026/04/17 11:34:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 11:34:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  AUC-ROC   : 0.8373  ← métrica principal
  Recall    : 0.7600  ← minimizar falsos negativos
  F1-score  : 0.2317
  Precision : 0.1367

              precision    recall  f1-score   support

   Sin ictus       0.98      0.75      0.85       947
       Ictus       0.14      0.76      0.23        50

    accuracy                           0.75       997
   macro avg       0.56      0.75      0.54       997
weighted avg       0.94      0.75      0.82       997

  ✓ Run registrado en MLflow

  RandomForest | dataset=full | SMOTE=False
✓ Dataset 'full': 4,981 filas
  Train: 3,984 filas | Test: 997 filas
  Stroke en train: 198 (5.0%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']


...


  AUC-ROC   : 0.8036  ← métrica principal
  Recall    : 0.1000  ← minimizar falsos negativos
  F1-score  : 0.1087
  Precision : 0.1190

              precision    recall  f1-score   support

   Sin ictus       0.95      0.96      0.96       947
       Ictus       0.12      0.10      0.11        50

    accuracy                           0.92       997
   macro avg       0.54      0.53      0.53       997
weighted avg       0.91      0.92      0.91       997

  ✓ Run registrado en MLflow

  LogisticRegression | dataset=adults | SMOTE=False
  Filtro adultos: 770 filas eliminadas
✓ Dataset 'adults': 4,211 filas
  Train: 3,368 filas | Test: 843 filas
  Stroke en train: 197 (5.8%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']


2026/04/17 11:34:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 11:34:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  AUC-ROC   : 0.8332  ← métrica principal
  Recall    : 0.8163  ← minimizar falsos negativos
  F1-score  : 0.2500
  Precision : 0.1476

              precision    recall  f1-score   support

   Sin ictus       0.98      0.71      0.82       794
       Ictus       0.15      0.82      0.25        49

    accuracy                           0.72       843
   macro avg       0.57      0.76      0.54       843
weighted avg       0.94      0.72      0.79       843

  ✓ Run registrado en MLflow

  RandomForest | dataset=adults | SMOTE=True
  Filtro adultos: 770 filas eliminadas
✓ Dataset 'adults': 4,211 filas
  Train: 3,368 filas | Test: 843 filas
  Stroke en train: 197 (5.8%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']


2026/04/17 11:34:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 11:34:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  AUC-ROC   : 0.7749  ← métrica principal
  Recall    : 0.1020  ← minimizar falsos negativos
  F1-score  : 0.1250
  Precision : 0.1613

              precision    recall  f1-score   support

   Sin ictus       0.95      0.97      0.96       794
       Ictus       0.16      0.10      0.12        49

    accuracy                           0.92       843
   macro avg       0.55      0.53      0.54       843
weighted avg       0.90      0.92      0.91       843

  ✓ Run registrado en MLflow
